In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. INITIALIZE SPARK
spark = SparkSession.builder \
    .appName("NYC_Taxi_Stage7_Classification") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. LOAD DATA AND PIPELINE
print("Loading data and Pipeline...")
train_df = spark.read.parquet("../data/processed/train.parquet")
val_df = spark.read.parquet("../data/processed/val.parquet")

# We unite train and val because TrainValidationSplit handles the split internally
training_data = train_df.union(val_df)

pipeline_model = PipelineModel.load("../models/preprocessing_pipeline")
print("Transforming data through the pipeline...")
prepared_data = pipeline_model.transform(training_data)

# 3. DEFINE EVALUATOR (Rubric: Use AUC-PR for imbalanced detection)
evaluator = BinaryClassificationEvaluator(
    labelCol="has_tip", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderPR" # Area Under Precision-Recall Curve
)

# *****************************************
# MODEL 1: LOGISTIC REGRESSION (Baseline)
# *****************************************
print("\n--- Training Model 1: Logistic Regression ---")
lr = LogisticRegression(featuresCol="features", labelCol="has_tip")

# Search Space: 2 x 2 = 4 combinations
lr_paramGrid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build())

lr_tvs = TrainValidationSplit(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=evaluator,
    trainRatio=0.8 # 80% of the internal data for training, 20% for validation
)

start_time = time.time()
lr_model = lr_tvs.fit(prepared_data)
lr_duration = time.time() - start_time

print(f"Logistic Regression Tuning completed in {lr_duration/60:.2f} minutes.")
print(f"Search space size: {len(lr_paramGrid)} combinations.")

best_lr = lr_model.bestModel
best_lr.write().overwrite().save("../models/best_logistic_regression")

# ********************************************
# MODEL 2: RANDOM FOREST CLASSIFIER (Ensemble)
# ********************************************
print("\n--- Training Model 2: Random Forest ---")
rf = RandomForestClassifier(featuresCol="features", labelCol="has_tip")

# Search Space: 2 x 2 = 4 combinations
rf_paramGrid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 10])
    .build())

rf_tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluator,
    trainRatio=0.8
)

start_time = time.time()
rf_model = rf_tvs.fit(prepared_data)
rf_duration = time.time() - start_time

print(f"Random Forest Tuning completed in {rf_duration/60:.2f} minutes.")
print(f"Search space size: {len(rf_paramGrid)} combinations.")

best_rf = rf_model.bestModel
best_rf.write().overwrite().save("../models/best_random_forest")

print("\nStage 7 (Classification) completed successfully. Best models saved to disk!")

26/05/27 02:15:39 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:15:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/27 02:15:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading data and Pipeline...


Transforming data through the pipeline...


...


[Stage 429:=========================================>             (25 + 8) / 33]




[Stage 429:================================================>      (29 + 4) / 33]




[Stage 429:=====================================================> (32 + 1) / 33]



Random Forest Tuning completed in 3.68 minutes.
Search space size: 4 combinations.



Stage 7 (Classification) completed successfully. Best models saved to disk!
